In [5]:

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/1e275a2206db45a08b19f2545e2ae8f5:46562652-ebad-4e2b-a27a-1531217eb801::"
)

In [6]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from qiskit_ibm_runtime import QiskitRuntimeService

# ============================================================
# CONFIGURATION
# ============================================================

JOBS = {
    "None": "dbm9mld8ft8s738uf8i0",
    "XpXm": "dbm9lvu74qds73drssp0",
    "XX": "dbm9val8mdtc73d7iuvg",
    "XY4": "dbma03u74qds73drticg",
    "Twirling": "dbm9mud8mdtc73d7iclg",
    "XpXm + Twirling": "dbm9kpu74qds73drsr20",
    "XX + Twirling": "dbm9uju74qds73drteug",
    "XY4 + Twirling": "dbma14m74qds73drtkng",
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_average(values):
    values = [
        v for v in values
        if v is not None and np.isfinite(v)
    ]
    return float(np.mean(values)) if values else np.nan


def get_parameter(parameters, name):
    for parameter in parameters:
        if parameter.name == name:
            return parameter.value
    return None


# ============================================================
# DATA COLLECTION
# ============================================================

summary_rows = []
detail_rows = []

for experiment_name, job_id in JOBS.items():

    print("\n" + "=" * 60)
    print(f"Experiment: {experiment_name}")
    print(f"Job ID: {job_id}")
    print("=" * 60)

    try:
        job = service.job(job_id)

        props = job.properties()

        if props is None:
            print("No historical calibration properties available.")
            continue

        backend = job.backend

        backend_name = (
            backend.name
            if hasattr(backend, "name")
            else str(backend)
        )

        calibration_time = props.last_update_date

        # ----------------------------------------------------
        # QUBIT CALIBRATION
        # ----------------------------------------------------

        readout_errors = []
        t1_values = []
        t2_values = []

        for qubit in range(len(props.qubits)):

            readout = props.readout_error(qubit)
            t1 = props.t1(qubit)
            t2 = props.t2(qubit)

            if readout is not None:
                readout_errors.append(readout)

            if t1 is not None:
                t1_values.append(t1)

            if t2 is not None:
                t2_values.append(t2)

            detail_rows.append({
                "Experiment": experiment_name,
                "Job ID": job_id,
                "Backend": backend_name,
                "Calibration Time": calibration_time,
                "Component": "Qubit",
                "Component Name": f"Qubit {qubit}",
                "Qubits": str([qubit]),
                "Readout Error": readout,
                "T1 (s)": t1,
                "T2 (s)": t2,
                "Gate Error": np.nan
            })

        # ----------------------------------------------------
        # GATE CALIBRATION
        # ----------------------------------------------------

        single_gate_errors = []
        two_gate_errors = []

        gate_type_errors = {}

        for gate in props.gates:

            gate_name = gate.gate
            qubits = gate.qubits

            error = get_parameter(
                gate.parameters,
                "gate_error"
            )

            if error is None:
                continue

            if len(qubits) == 1:
                single_gate_errors.append(error)

            elif len(qubits) == 2:
                two_gate_errors.append(error)

            gate_type_errors.setdefault(
                gate_name, []
            ).append(error)

            detail_rows.append({
                "Experiment": experiment_name,
                "Job ID": job_id,
                "Backend": backend_name,
                "Calibration Time": calibration_time,
                "Component": "Gate",
                "Component Name": gate_name,
                "Qubits": str(qubits),
                "Readout Error": np.nan,
                "T1 (s)": np.nan,
                "T2 (s)": np.nan,
                "Gate Error": error
            })

        # ----------------------------------------------------
        # SUMMARY
        # ----------------------------------------------------

        row = {
            "Experiment": experiment_name,
            "Job ID": job_id,
            "Backend": backend_name,
            "Calibration Time": calibration_time,

            "Average Readout Error":
                get_average(readout_errors),

            "Average Single-Qubit Gate Error":
                get_average(single_gate_errors),

            "Average Two-Qubit Gate Error":
                get_average(two_gate_errors),

            "Average T1 (s)":
                get_average(t1_values),

            "Average T2 (s)":
                get_average(t2_values),

            "Number of Qubits":
                len(props.qubits),

            "Number of Single-Qubit Gates":
                len(single_gate_errors),

            "Number of Two-Qubit Gates":
                len(two_gate_errors)
        }

        # Individual gate-type averages
        for gate_name, errors in gate_type_errors.items():

            row[f"Average {gate_name} Error"] = (
                get_average(errors)
            )

        summary_rows.append(row)

        print("Successfully retrieved calibration data.")
        print(
            f"Readout error: "
            f"{row['Average Readout Error']:.6f}"
        )
        print(
            f"Single-qubit error: "
            f"{row['Average Single-Qubit Gate Error']:.6f}"
        )
        print(
            f"Two-qubit error: "
            f"{row['Average Two-Qubit Gate Error']:.6f}"
        )

    except Exception as e:

        print(f"ERROR processing {experiment_name}: {e}")


# ============================================================
# EXPORT CSV FILES
# ============================================================

summary_df = pd.DataFrame(summary_rows)
details_df = pd.DataFrame(detail_rows)

if summary_df.empty:
    raise RuntimeError(
        "No calibration data was retrieved. "
        "Check your job IDs and IBM Runtime access."
    )

summary_df.to_csv(
    "hardware_error_summary.csv",
    index=False
)

details_df.to_csv(
    "hardware_error_details.csv",
    index=False
)

print("\nCSV FILES SAVED")
print("hardware_error_summary.csv")
print("hardware_error_details.csv")


# ============================================================
# GRAPH GENERATION
# ============================================================

metrics = [
    "Average Readout Error",
    "Average Single-Qubit Gate Error",
    "Average Two-Qubit Gate Error"
]

for metric in metrics:

    if metric not in summary_df.columns:
        continue

    if summary_df[metric].isna().all():
        print(f"Skipping {metric}: no data available.")
        continue

    plt.figure(figsize=(12, 6))

    plt.bar(
        summary_df["Experiment"],
        summary_df[metric]
    )

    plt.title(
        f"{metric} by Experiment"
    )

    plt.ylabel("Error Probability")
    plt.xlabel("Experiment")

    plt.xticks(
        rotation=35,
        ha="right"
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.tight_layout()

    filename = (
        metric.lower()
        .replace(" ", "_")
        + ".png"
    )

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved graph: {filename}")


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("HARDWARE ERROR ANALYSIS COMPLETE")
print("=" * 70)

print(
    summary_df.to_string(index=False)
)

print("\nGenerated files:")
print("1. hardware_error_summary.csv")
print("2. hardware_error_details.csv")
print("3. average_readout_error.png")
print("4. average_single-qubit_gate_error.png")
print("5. average_two-qubit_gate_error.png")



Experiment: None
Job ID: dbm9mld8ft8s738uf8i0
ERROR processing None: "Couldn't find the property 'T2' for qubit 72."

Experiment: XpXm
Job ID: dbm9lvu74qds73drssp0
ERROR processing XpXm: "Couldn't find the property 'T2' for qubit 72."

Experiment: XX
Job ID: dbm9val8mdtc73d7iuvg
ERROR processing XX: "Couldn't find the property 'T2' for qubit 72."

Experiment: XY4
Job ID: dbma03u74qds73drticg
ERROR processing XY4: "Couldn't find the property 'T2' for qubit 72."

Experiment: Twirling
Job ID: dbm9mud8mdtc73d7iclg
ERROR processing Twirling: "Couldn't find the property 'T2' for qubit 72."

Experiment: XpXm + Twirling
Job ID: dbm9kpu74qds73drsr20
ERROR processing XpXm + Twirling: "Couldn't find the property 'T2' for qubit 72."

Experiment: XX + Twirling
Job ID: dbm9uju74qds73drteug
ERROR processing XX + Twirling: "Couldn't find the property 'T2' for qubit 72."

Experiment: XY4 + Twirling
Job ID: dbma14m74qds73drtkng
ERROR processing XY4 + Twirling: "Couldn't find the property 'T2' for qubit

RuntimeError: No calibration data was retrieved. Check your job IDs and IBM Runtime access.